In [0]:
df = spark.read.format('csv').option('header','true').csv('/Workspace/MyWorkSpace/orders.csv')

In [0]:
df.show()

+----------+------------+-----------+------+--------+-----+-------+
|Order Date|Product Name|   Category|Region|Quantity|Sales| Profit|
+----------+------------+-----------+------+--------+-----+-------+
|2024-12-31|     Printer|     Office| North|       4| 3640| 348.93|
|2022-11-27|       Mouse|Accessories|  East|       7| 1197| 106.53|
|2022-05-11|      Tablet|Electronics| South|       5| 5865| 502.73|
|2024-03-16|       Mouse|Accessories| South|       2|  786| 202.87|
|2022-09-10|       Mouse|Accessories|  West|       1|  509| 103.28|
|2023-12-01|      Camera|Electronics|  West|       1|  524| 106.35|
|2023-10-09|  Headphones|Accessories| North|       7| 6167|1027.98|
|2022-01-14|      Camera|Electronics| South|       7| 3059|  873.5|
|2022-04-02|  Smartwatch|Electronics|  East|       9| 5526| 595.28|
|2024-10-22|     Printer|     Office| South|       8|  672| 186.37|
|2023-12-04|     Monitor|Accessories| South|       6| 7074|1357.68|
|2022-09-27|  Smartwatch|Electronics|  East|    

In [0]:
df = df.withColumnsRenamed({
    'Order Date': 'Order_Date',
    'Product Name': 'Product_Name'
})

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .option('header','true')\
    .saveAsTable('orders')

In [0]:
%sql
select * from orders version as of 0 limit 5;

Order_Date,Product_Name,Category,Region,Quantity,Sales,Profit
2024-12-31,Printer,Office,North,4,3640,348.93
2022-11-27,Mouse,Accessories,East,7,1197,106.53
2022-05-11,Tablet,Electronics,South,5,5865,502.73
2024-03-16,Mouse,Accessories,South,2,786,202.87
2022-09-10,Mouse,Accessories,West,1,509,103.28


In [0]:
%sql
ALTER TABLE orders
ADD COLUMN profit_decimal DECIMAL(10,2);

In [0]:
%sql
UPDATE orders
SET profit_decimal = try_cast(Profit AS DECIMAL(10,2));

num_affected_rows
3500


In [0]:
%sql
describe history orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-05-08T09:18:53Z,141741144301471,chauhansk1712@gmail.com,UPDATE,Map(predicate -> []),null,List(935789528312351),0508-084427-95ywfgd7,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 49321, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1594, numDeletionVectorsUpdated -> 0, scanTimeMs -> 450, numAddedFiles -> 1, numUpdatedRows -> 3500, numAddedBytes -> 49321, rewriteTimeMs -> 1143)",null,Databricks-Runtime/17.3.x-scala2.13
2,2026-05-08T09:18:07Z,141741144301471,chauhansk1712@gmail.com,UPDATE,Map(predicate -> []),null,List(935789528312351),0508-084427-95ywfgd7,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 39009, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2961, numDeletionVectorsUpdated -> 0, scanTimeMs -> 899, numAddedFiles -> 1, numUpdatedRows -> 3500, numAddedBytes -> 49321, rewriteTimeMs -> 2029)",null,Databricks-Runtime/17.3.x-scala2.13
1,2026-05-08T09:16:21Z,141741144301471,chauhansk1712@gmail.com,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""profit_decimal"",""type"":""decimal(10,2)"",""nullable"":true,""metadata"":{}}}])",null,List(935789528312351),0508-084427-95ywfgd7,0,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-scala2.13
0,2026-05-08T09:14:57Z,141741144301471,chauhansk1712@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(935789528312351),0508-084427-95ywfgd7,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3500, numOutputBytes -> 39009)",null,Databricks-Runtime/17.3.x-scala2.13


In [0]:
%sql
describe detail orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,30364dd2-3e43-44cb-90ef-7b8307535636,az_databricks_demo.default.orders,null,abfss://unity-catalog-storage@dbstoragev5rmljgd5cp36.dfs.core.windows.net/7405607748564183/__unitystorage/catalogs/2f319b98-0b48-44f9-8f58-aa924ed17462/tables/6f80ff4d-e85b-4532-9388-250586e87235,2026-05-08T09:14:42.116Z,2026-05-08T09:18:53Z,List(),List(),1,49321,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
delete from orders where Quantity < 3;

num_affected_rows
805


In [0]:
%sql
update orders
set Sales = Sales + 10
where Region = 'North';

num_affected_rows
641


In [0]:
%sql
update orders
set Sales = Sales + 15
where Region = 'East';

num_affected_rows
655


In [0]:
%sql
describe history orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-05-08T09:25:22Z,141741144301471,chauhansk1712@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(935789528312351),0508-084427-95ywfgd7,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 51772, p25FileSize -> 39561, numDeletionVectorsRemoved -> 1, minFileSize -> 39561, numAddedFiles -> 1, maxFileSize -> 39561, p75FileSize -> 39561, p50FileSize -> 39561, numAddedBytes -> 39561)",null,Databricks-Runtime/17.3.x-scala2.13
8,2026-05-08T09:25:16Z,141741144301471,chauhansk1712@gmail.com,UPDATE,"Map(predicate -> [""(Region#4418 = East)""])",null,List(935789528312351),0508-084427-95ywfgd7,7,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4469, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1944, numAddedFiles -> 1, numUpdatedRows -> 655, numAddedBytes -> 12211, rewriteTimeMs -> 2524)",null,Databricks-Runtime/17.3.x-scala2.13
7,2026-05-08T09:24:22Z,141741144301471,chauhansk1712@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(935789528312351),0508-084427-95ywfgd7,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 51173, p25FileSize -> 39561, numDeletionVectorsRemoved -> 1, minFileSize -> 39561, numAddedFiles -> 1, maxFileSize -> 39561, p75FileSize -> 39561, p50FileSize -> 39561, numAddedBytes -> 39561)",null,Databricks-Runtime/17.3.x-scala2.13
6,2026-05-08T09:24:16Z,141741144301471,chauhansk1712@gmail.com,UPDATE,"Map(predicate -> [""(Region#3442 = North)""])",null,List(935789528312351),0508-084427-95ywfgd7,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4910, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2148, numAddedFiles -> 1, numUpdatedRows -> 641, numAddedBytes -> 12035, rewriteTimeMs -> 2761)",null,Databricks-Runtime/17.3.x-scala2.13
5,2026-05-08T09:20:14Z,141741144301471,chauhansk1712@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(935789528312351),0508-084427-95ywfgd7,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 49321, p25FileSize -> 39138, numDeletionVectorsRemoved -> 1, minFileSize -> 39138, numAddedFiles -> 1, maxFileSize -> 39138, p75FileSize -> 39138, p50FileSize -> 39138, numAddedBytes -> 39138)",null,Databricks-Runtime/17.3.x-scala2.13
4,2026-05-08T09:20:06Z,141741144301471,chauhansk1712@gmail.com,DELETE,"Map(predicate -> [""(cast(Quantity#2638 as bigint) < 3)""])",null,List(935789528312351),0508-084427-95ywfgd7,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 8564, numDeletionVectorsUpdated -> 0, numDeletedRows -> 805, scanTimeMs -> 3830, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 4714)",null,Databricks-Runtime/17.3.x-scala2.13
3,2026-05-08T09:18:53Z,141741144301471,chauhansk1712@gmail.com,UPDATE,Map(predicate -> []),null,List(935789528312351),0508-084427-95ywfgd7,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 49321, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1594, numDeletionVectorsUpdated -> 0, scanTimeMs -> 450, numAddedFiles -> 1, numUpdatedRows -> 3500, numAddedBytes -> 49321, rewriteTimeMs -> 1143)",null,Databricks-Runtime/17.3.x-scala2.13
2,2026-05-08T09:18:07Z,141741144301471,chauhansk1712@gmail.com,UPDATE,Map(predic

In [0]:
%sql
describe detail orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,30364dd2-3e43-44cb-90ef-7b8307535636,az_databricks_demo.default.orders,null,abfss://unity-catalog-storage@dbstoragev5rmljgd5cp36.dfs.core.windows.net/7405607748564183/__unitystorage/catalogs/2f319b98-0b48-44f9-8f58-aa924ed17462/tables/6f80ff4d-e85b-4532-9388-250586e87235,2026-05-08T09:14:42.116Z,2026-05-08T09:25:22Z,List(),List(),1,39561,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT * FROM orders VERSION AS OF 0 limit 5;

Order_Date,Product_Name,Category,Region,Quantity,Sales,Profit
2024-12-31,Printer,Office,North,4,3640,348.93
2022-11-27,Mouse,Accessories,East,7,1197,106.53
2022-05-11,Tablet,Electronics,South,5,5865,502.73
2024-03-16,Mouse,Accessories,South,2,786,202.87
2022-09-10,Mouse,Accessories,West,1,509,103.28


In [0]:
spark.conf.set(
    "spark.databricks.delta.retentionDurationCheck.enabled",
    "false"
)

In [0]:
%sql
VACUUM orders RETAIN 0 HOURS;

path
abfss://unity-catalog-storage@dbstoragev5rmljgd5cp36.dfs.core.windows.net/7405607748564183/__unitystorage/catalogs/2f319b98-0b48-44f9-8f58-aa924ed17462/tables/6f80ff4d-e85b-4532-9388-250586e87235


In [0]:
%sql
describe detail orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,30364dd2-3e43-44cb-90ef-7b8307535636,az_databricks_demo.default.orders,null,abfss://unity-catalog-storage@dbstoragev5rmljgd5cp36.dfs.core.windows.net/7405607748564183/__unitystorage/catalogs/2f319b98-0b48-44f9-8f58-aa924ed17462/tables/6f80ff4d-e85b-4532-9388-250586e87235,2026-05-08T09:14:42.116Z,2026-05-08T09:34:39Z,List(),List(),1,39561,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT * FROM orders VERSION AS OF 0 limit 5;

org.apache.spark.SparkException: [FAILED_READ_FILE.DBR_FILE_NOT_EXIST] Error while reading file abfss://REDACTED_CREDENTIALS(686ca54d)@dbstoragev5rmljgd5cp36.dfs.core.windows.net/7405607748564183/__unitystorage/catalogs/2f319b98-0b48-44f9-8f58-aa924ed17462/tables/6f80ff4d-e85b-4532-9388-250586e87235/part-00000-2035ec00-cb00-42a4-a359-cb36c1a1d935-c000.zstd.parquet. [DELTA_FILE_NOT_FOUND_DETAILED] File abfss://unity-catalog-storage@dbstoragev5rmljgd5cp36.dfs.core.windows.net/7405607748564183/__unitystorage/catalogs/2f319b98-0b48-44f9-8f58-aa924ed17462/tables/6f80ff4d-e85b-4532-9388-250586e87235/part-00000-2035ec00-cb00-42a4-a359-cb36c1a1d935-c000.zstd.parquet referenced in the transaction log cannot be found. This occurs when data has been manually deleted from the file system rather than using the table `DELETE` statement. For more information, see https://docs.microsoft.com/azure/databricks/delta/delta-intro#frequently-asked-questions
 SQLSTATE: KD001
	at org.apache.spark.sql.errors.Q